# RNN MFCC-40
Trains multilabel and binary UUV BiLSTMs for normal, M-filtered, and W-filtered MFCC-40 data from the mfcc12 Kaggle archive.

In [ ]:
import sys
from pathlib import Path

PIPELINE_GITHUB_RAW_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/spectrogram_pipeline.py"  # Optional raw GitHub URL for spectrogram_pipeline.py
MODULE_FILE = "spectrogram_pipeline.py"
candidate_dirs = [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/drive/MyDrive/STUDA/src")]
module_path = next((directory / MODULE_FILE for directory in candidate_dirs if (directory / MODULE_FILE).exists()), None)

if module_path is None and PIPELINE_GITHUB_RAW_URL:
    import urllib.request
    module_path = Path("/content") / MODULE_FILE
    urllib.request.urlretrieve(PIPELINE_GITHUB_RAW_URL, module_path)

if module_path is None or not module_path.exists():
    raise FileNotFoundError(f"{MODULE_FILE} was not found. Sync, upload, mount, or set PIPELINE_GITHUB_RAW_URL.")

sys.path.insert(0, str(module_path.parent))
print(f"Using pipeline module: {module_path}")


In [ ]:
import pandas as pd
import tensorflow as tf
from IPython.display import display
from google.colab import files, userdata

from spectrogram_pipeline import (
    build_rnn_models_for_variants, evaluate_models_for_variants, extract_zip,
    get_rnn_callbacks, plot_training_histories, prepare_mfcc_dataset_variants,
    save_artifacts, train_models_for_variants, zip_artifacts,
)

USE_TPU = True
if USE_TPU:
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
    except (ValueError, RuntimeError):
        strategy = tf.distribute.get_strategy()
else:
    strategy = tf.distribute.get_strategy()
print(f"Replicas in sync: {strategy.num_replicas_in_sync}")


In [ ]:
DATASET_KEY = "mfcc40"
DATASET_LABEL = "MFCC-40"
DATASET_SLUG = "pawedyrda/mfcc12"
ARCHIVE_PATH = Path("/content/mfcc12.zip")
EPOCHS = 50
BATCH_SIZE = 64


In [ ]:
kaggle_token = userdata.get("Kaggle")
if kaggle_token is None:
    raise ValueError("Missing Colab secret named 'Kaggle'.")

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / "access_token"
token_path.write_text(kaggle_token)
token_path.chmod(0o600)


In [ ]:
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
DATA_PATH = extract_zip(ARCHIVE_PATH, "/content")
print(f"Dataset extracted to: {DATA_PATH}")


In [ ]:
variants = prepare_mfcc_dataset_variants(DATA_PATH)
print("Normal train shape:", variants.normal.train_data.shape)
print("M train shape:", variants.m.train_data.shape)
print("W train shape:", variants.w.train_data.shape)


In [ ]:
with strategy.scope():
    multilabel_models = build_rnn_models_for_variants(variants, model_type="multilabel")

multilabel_histories = train_models_for_variants(
    multilabel_models, variants, model_type="multilabel", epochs=EPOCHS,
    batch_size=BATCH_SIZE, callback_factory=get_rnn_callbacks,
)


In [ ]:
with strategy.scope():
    binary_models = build_rnn_models_for_variants(variants, model_type="binary")

binary_histories = train_models_for_variants(
    binary_models, variants, model_type="binary", epochs=EPOCHS,
    batch_size=BATCH_SIZE, callback_factory=get_rnn_callbacks,
)


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([
    multilabel_results.assign(task="multilabel"),
    binary_results.assign(task="binary"),
], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])


In [ ]:
plot_training_histories(multilabel_histories, f"Multilabel RNN Training Curves - {DATASET_LABEL}")
plot_training_histories(binary_histories, f"Binary RNN Training Curves - {DATASET_LABEL}")

save_dir = save_artifacts(
    f"/content/saved_artifacts/rnn_{DATASET_KEY}", DATASET_KEY, multilabel_models, binary_models,
    multilabel_histories, binary_histories, multilabel_results, binary_results,
)
comparison_results.to_csv(save_dir / f"rnn_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/rnn_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
